In [ ]:
import argparse
import os
MODEL_DIR = "../models/Qwen3.5-2B"

# parser = argparse.ArgumentParser()
# parser.add_argument("--attn_implementation", type=str, default="eager")
# parser.add_argument("--dtype", type=str, default="bfloat16", choices=["bfloat16", "float32"])
# parser.add_argument("--out", type=str, default="../dumps/oracle_bf16.safetensors")
# args = parser.parse_args()

out_file = "../dumps/oracle_bf16.safetensors"
os.makedirs(os.path.dirname(out_file), exist_ok=True)

prompt = "Hello, introduce yourself. by letting me know all the features that you have and can do. also let me know the best way to use you."

TAPS = {}

In [2]:
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer
from safetensors.torch import save_file, load_file


tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

d:\AI\Inferencing\deltaforge\python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokens = tokenizer(prompt, return_tensors="pt")
print(tokens)

{'input_ids': tensor([[ 9419,    11, 18553,  5941,    13,   539, 19581,   728,  1366,   660,
           279,  4275,   421,   488,   599,   321,   628,   635,    13,  1048,
          1042,   728,  1366,   279,  1786,  1560,   310,   958,   488,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}


In [4]:
model = AutoModelForImageTextToText.from_pretrained(
        MODEL_DIR, dtype=torch.bfloat16, low_cpu_mem_usage=True,
        attn_implementation="eager"
    )
model.eval()
print(model.config._attn_implementation)

Loading weights: 100%|██████████| 617/617 [00:00<00:00, 4881.30it/s]

eager


In [5]:
def tap(name, t):
    if name in TAPS:
        raise ValueError(f"TAP {name} already exists")

    if isinstance(t, tuple):
        t = t[0]
        
    assert t.shape[0] == 1, f"{name}: expected batch 1, got {tuple(t.shape)}"
    t = t[0]
    TAPS[name] = t.detach().clone().contiguous()


In [6]:
def make_out_hook(name):
    def hook(module, args, output):
        tap(name, output)
    
    return hook

def make_in_hook(name):
    def hook(module, args):
        tap(name, args[0])
    return hook

def make_gate_hook(name, head_dim):
    def hook(module, args, output):
        input_shape = output.shape[:-1]
        _, gate = torch.chunk(output.view(*input_shape, -1, head_dim * 2), 2, dim=-1)
        tap(name, gate.reshape(*input_shape, -1))
        
    return hook

In [7]:
# --- step 4: wrappers for module-level functions (no module to hook) ---------------
# forward() looks these names up in the module's globals at call time, so replacing
# mq.<name> is seen by the next forward. unpatch() puts the originals back.
import inspect
import transformers.models.qwen3_5.modeling_qwen3_5 as mq

WRAPPED = ["apply_rotary_pos_emb", "eager_attention_forward", "torch_chunk_gated_delta_rule"]
if "ORIG" not in globals():  # capture once: re-running this cell must never capture a wrapper
    ORIG = {n: getattr(mq, n) for n in WRAPPED}

CURRENT = {"layer": None}  # set by the self_attn pre-hook; RoPE has no layer_idx of its own
CALLS = {n: 0 for n in WRAPPED}

def make_layer_marker(i):
    def hook(module, args):
        CURRENT["layer"] = i
    return hook

def rope_wrapper(*args, **kwargs):
    q_out, k_out = ORIG["apply_rotary_pos_emb"](*args, **kwargs)
    CALLS["apply_rotary_pos_emb"] += 1
    p = f"L{CURRENT['layer']:02d}"
    tap(f"{p}.q_roped", q_out)  # [1, 8, T, 256] -> [8, T, 256]  (heads before tokens)
    tap(f"{p}.k_roped", k_out)  # [1, 2, T, 256] -> [2, T, 256]
    return q_out, k_out

def eager_wrapper(module, *args, **kwargs):
    attn_output, attn_weights = ORIG["eager_attention_forward"](module, *args, **kwargs)
    CALLS["eager_attention_forward"] += 1
    tap(f"L{module.layer_idx:02d}.core_out", attn_output)  # [1, T, 8, 256] -> [T, 8, 256]
    return attn_output, attn_weights

def gdn_chunk_counter(*args, **kwargs):  # no tap yet (06/07); proves this path ran 18x
    CALLS["torch_chunk_gated_delta_rule"] += 1
    return ORIG["torch_chunk_gated_delta_rule"](*args, **kwargs)

def patch():
    mq.apply_rotary_pos_emb = rope_wrapper
    mq.eager_attention_forward = eager_wrapper
    mq.torch_chunk_gated_delta_rule = gdn_chunk_counter

def unpatch():
    for n, f in ORIG.items():
        setattr(mq, n, f)

# --- step 5a: which GDN implementation was resolved at import time ------------------
# The decorator (integrations/hub_kernels.py:764) builds `wrapped`, which calls a closure
# variable `implementation`. functools.wraps copies the torch function's name onto it,
# so read the closure, not __name__.
GDN_IMPL = {}
for n in ["torch_chunk_gated_delta_rule", "torch_recurrent_gated_delta_rule",
          "causal_conv1d_fn", "causal_conv1d_update"]:
    fn = ORIG.get(n, getattr(mq, n))
    impl = inspect.getclosurevars(fn).nonlocals.get("implementation")
    assert impl is not None, f"{n}: no `implementation` in closure; decorator changed, inspect by hand"
    GDN_IMPL[n] = f"{impl.__module__}.{impl.__qualname__}"
    print(f"{n:34s} -> {GDN_IMPL[n]}")
    assert impl.__module__ == mq.__name__, f"{n} is NOT the torch fallback; dump is not the math we reimplement"

# --- step 5b: end-of-prompt cache -------------------------------------------------
def tap_cache(cache):
    layer_types = model.model.language_model.config.layer_types
    for i, lt in enumerate(layer_types):
        L, p = cache.layers[i], f"L{i:02d}.cache"
        if lt == "full_attention":
            tap(f"{p}.k", L.keys)    # [1, 2, T, 256] -> [2, T, 256], post-norm post-RoPE
            tap(f"{p}.v", L.values)  # [1, 2, T, 256] -> [2, T, 256]
        else:
            tap(f"{p}.conv", L.conv_states[0])
            tap(f"{p}.recurrent", L.recurrent_states[0])

torch_chunk_gated_delta_rule       -> transformers.models.qwen3_5.modeling_qwen3_5.torch_chunk_gated_delta_rule
torch_recurrent_gated_delta_rule   -> transformers.models.qwen3_5.modeling_qwen3_5.torch_recurrent_gated_delta_rule
causal_conv1d_fn                   -> transformers.models.qwen3_5.modeling_qwen3_5.causal_conv1d_fn
causal_conv1d_update               -> transformers.models.qwen3_5.modeling_qwen3_5.causal_conv1d_update


In [8]:
def register_hooks():
    layers = model.model.language_model.layers
    handles = []
    lm = model.model.language_model
    handles.append(lm.embed_tokens.register_forward_hook(make_out_hook("embed")))
    handles.append(lm.norm.register_forward_hook(make_out_hook("final_norm")))
    for i, layer in enumerate(layers):
        LIdx_prefix = f'L{i:02d}'
        handles.append(layer.register_forward_pre_hook(make_in_hook(f"{LIdx_prefix}.in")))
        handles.append(layer.register_forward_hook(make_out_hook(f"{LIdx_prefix}.out")))
        
        if not hasattr(layer, "self_attn"): continue
        
        self_attn = layer.self_attn
        handles += [
            self_attn.register_forward_pre_hook(make_layer_marker(i)),  # step 4: tells rope_wrapper the layer
            layer.input_layernorm.register_forward_hook(make_out_hook(f"{LIdx_prefix}.input_norm")),
            self_attn.q_norm.register_forward_pre_hook(make_in_hook(f"{LIdx_prefix}.q")),
            self_attn.q_proj.register_forward_hook(make_gate_hook(f"{LIdx_prefix}.gate", self_attn.head_dim)),
            self_attn.k_norm.register_forward_pre_hook(make_in_hook(f"{LIdx_prefix}.k")),
            self_attn.v_proj.register_forward_hook(make_out_hook(f"{LIdx_prefix}.v")),
            self_attn.q_norm.register_forward_hook(make_out_hook(f"{LIdx_prefix}.q_normed")),
            self_attn.k_norm.register_forward_hook(make_out_hook(f"{LIdx_prefix}.k_normed")),
            self_attn.o_proj.register_forward_pre_hook(make_in_hook(f"{LIdx_prefix}.gated")),
            self_attn.register_forward_hook(make_out_hook(f"{LIdx_prefix}.mixer_out")),
            layer.post_attention_layernorm.register_forward_pre_hook(make_in_hook(f"{LIdx_prefix}.resid1")),
            layer.post_attention_layernorm.register_forward_hook(make_out_hook(f"{LIdx_prefix}.post_norm")),
            layer.mlp.register_forward_hook(make_out_hook(f"{LIdx_prefix}.mlp_out")),
            
            self_attn.q_proj.register_forward_hook(make_out_hook(f"{LIdx_prefix}.q_proj")),
        ]

    return handles

In [9]:
# RESET TAPS and handles
# TAPS.clear()
# for h in handles: h.remove()

In [10]:
TAPS.clear()
CURRENT["layer"] = None
for n in CALLS: CALLS[n] = 0

handles = register_hooks()  # outside try: a half-registered set must not leave stale handles
try:
    patch()
    with torch.inference_mode():
        out = model(input_ids=tokens["input_ids"], use_cache=True)
finally:
    unpatch()
    for h in handles: h.remove()

tap("ids", tokens["input_ids"])
tap("logits", out.logits)
tap_cache(out.past_key_values)

print(len(TAPS), "taps (expect 196)")
print(CALLS, "(expect 6 / 6 / 18)")

196 taps (expect 196)
{'apply_rotary_pos_emb': 6, 'eager_attention_forward': 6, 'torch_chunk_gated_delta_rule': 18} (expect 6 / 6 / 18)


In [11]:
print("embed == L00.in", torch.equal(TAPS["embed"], TAPS["L00.in"]))
for i in range(23):
    print(f"L{i:02d}.out == L{i+1:02d}.in", torch.equal(TAPS[f"L{i:02d}.out"], TAPS[f"L{i+1:02d}.in"]))

embed == L00.in True
L00.out == L01.in True
L01.out == L02.in True
L02.out == L03.in True
L03.out == L04.in True
L04.out == L05.in True
L05.out == L06.in True
L06.out == L07.in True
L07.out == L08.in True
L08.out == L09.in True
L09.out == L10.in True
L10.out == L11.in True
L11.out == L12.in True
L12.out == L13.in True
L13.out == L14.in True
L14.out == L15.in True
L15.out == L16.in True
L16.out == L17.in True
L17.out == L18.in True
L18.out == L19.in True
L19.out == L20.in True
L20.out == L21.in True
L21.out == L22.in True
L22.out == L23.in True


In [12]:
torch.equal(TAPS["final_norm"], TAPS["L23.out"])

False

In [13]:
# --- checks for steps 3-5 -----------------------------------------------------------
# OK = expected equal and is; FAIL = expected equal, isn't.
# XFAIL = negative control differs (good); BAD = negative control matched (test is blind).
# Modules are re-run on [None]-batched inputs so they see the exact shapes HF used.
lm = model.model.language_model
T = TAPS["ids"].shape[0]
fails = []

def check(label, same, expect=True):
    status = ("OK" if same else "FAIL") if expect else ("BAD" if same else "XFAIL")
    if same != expect: fails.append(label)
    print(f"{status:5s} {label}")

check(f"call counts {CALLS}", CALLS == {"apply_rotary_pos_emb": 6, "eager_attention_forward": 6,
                                        "torch_chunk_gated_delta_rule": 18})

with torch.inference_mode():
    for i, layer in enumerate(lm.layers):
        p = f"L{i:02d}"
        t = lambda n: TAPS[f"{p}.{n}"]
        if not hasattr(layer, "self_attn"):
            check(f"{p} GDN cache present", f"{p}.cache.conv" in TAPS and f"{p}.cache.recurrent" in TAPS)
            continue
        a = layer.self_attn
        # step 3: residual adds + module re-runs on captured inputs
        check(f"{p} resid1 == in + mixer_out", torch.equal(t("resid1"), t("in") + t("mixer_out")))
        check(f"{p} out == resid1 + mlp_out", torch.equal(t("out"), t("resid1") + t("mlp_out")))
        check(f"{p} mixer_out == o_proj(gated)", torch.equal(t("mixer_out"), a.o_proj(t("gated")[None])[0]))
        check(f"{p} q_normed == q_norm(q)", torch.equal(t("q_normed"), a.q_norm(t("q")[None])[0]))
        check(f"{p} k_normed == k_norm(k)", torch.equal(t("k_normed"), a.k_norm(t("k")[None])[0]))
        check(f"{p} post_norm == post_attention_layernorm(resid1)",
              torch.equal(t("post_norm"), layer.post_attention_layernorm(t("resid1")[None])[0]))
        check(f"{p} input_norm == input_layernorm(in)",
              torch.equal(t("input_norm"), layer.input_layernorm(t("in")[None])[0]))
        # negative control: naive first-half/second-half split must NOT be the gate
        check(f"{p} naive gate q_proj[..., 2048:] != gate", torch.equal(t("q_proj")[..., 2048:], t("gate")), expect=False)
        # step 4: shapes + gate correctness (exact only if the per-head split is right)
        check(f"{p} q_roped/k_roped/core_out shapes",
              (tuple(t("q_roped").shape), tuple(t("k_roped").shape), tuple(t("core_out").shape))
              == ((8, T, 256), (2, T, 256), (T, 8, 256)))
        check(f"{p} gated == core_out.flat * sigmoid(gate)",
              torch.equal(t("gated"), (t("core_out").reshape(1, T, -1) * torch.sigmoid(t("gate")[None]))[0]))
        # step 5: KV cache holds post-RoPE k and raw v
        check(f"{p} cache.k == k_roped", torch.equal(t("cache.k"), t("k_roped")))
        check(f"{p} cache.v == v as [2, T, 256]", torch.equal(t("cache.v"), t("v").view(T, 2, 256).transpose(0, 1)))

print(f"\n{len(fails)} failing:", fails)
assert not fails, fails

OK    call counts {'apply_rotary_pos_emb': 6, 'eager_attention_forward': 6, 'torch_chunk_gated_delta_rule': 18}
OK    L00 GDN cache present
OK    L01 GDN cache present
OK    L02 GDN cache present
OK    L03 resid1 == in + mixer_out
OK    L03 out == resid1 + mlp_out
OK    L03 mixer_out == o_proj(gated)
OK    L03 q_normed == q_norm(q)
OK    L03 k_normed == k_norm(k)
OK    L03 post_norm == post_attention_layernorm(resid1)
OK    L03 input_norm == input_layernorm(in)
XFAIL L03 naive gate q_proj[..., 2048:] != gate
OK    L03 q_roped/k_roped/core_out shapes
OK    L03 gated == core_out.flat * sigmoid(gate)
OK    L03 cache.k == k_roped
OK    L03 cache.v == v as [2, T, 256]
OK    L04 GDN cache present
OK    L05 GDN cache present
OK    L06 GDN cache present
OK    L07 resid1 == in + mixer_out
OK    L07 out == resid1 + mlp_out
OK    L07 mixer_out == o_proj(gated)
OK    L07 q_normed == q_norm(q)
OK    L07 k_normed == k_norm(k)
OK    L07 post_norm == post_attention_layernorm(resid1)
OK    L07 input_no

In [14]:
import transformers

# manifest
for k, v in TAPS.items():
    print(f"{k:20s} {str(tuple(v.shape)):18s} {v.dtype}")

# save
save_file(TAPS, out_file, metadata={
    "prompt": prompt,
    "dtype": "bf16",
    "attn_implementation": model.config._attn_implementation,
    "gdn_impl": "; ".join(f"{k}={v}" for k, v in GDN_IMPL.items()),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
})

# reload: same keys, every tensor bit-exact
loaded = load_file(out_file)
assert loaded.keys() == TAPS.keys(), set(loaded.keys()) ^ set(TAPS.keys())
bad = [k for k in TAPS if loaded[k].dtype != TAPS[k].dtype or not torch.equal(loaded[k], TAPS[k])]
assert not bad, bad
print(f"reload OK: {len(loaded)} tensors, {os.path.getsize(out_file) / 2**20:.1f} MiB -> {out_file}")

embed                (30, 2048)         torch.bfloat16
L00.in               (30, 2048)         torch.bfloat16
L00.out              (30, 2048)         torch.bfloat16
L01.in               (30, 2048)         torch.bfloat16
L01.out              (30, 2048)         torch.bfloat16
L02.in               (30, 2048)         torch.bfloat16
L02.out              (30, 2048)         torch.bfloat16
L03.in               (30, 2048)         torch.bfloat16
L03.input_norm       (30, 2048)         torch.bfloat16
L03.gate             (30, 2048)         torch.bfloat16
L03.q_proj           (30, 4096)         torch.bfloat16
L03.q                (30, 8, 256)       torch.bfloat16
L03.q_normed         (30, 8, 256)       torch.bfloat16
L03.k                (30, 2, 256)       torch.bfloat16
L03.k_normed         (30, 2, 256)       torch.bfloat16
L03.v                (30, 512)          torch.bfloat16
L03.q_roped          (8, 30, 256)       torch.bfloat16
L03.k_roped          (2, 30, 256)       torch.bfloat16
L03.core_o